# **Text Classification: -**



In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("ashfakyeafi/spam-email-classification")

print("Path to dataset files:", path)

/Users/yuvrajbhatkariya/data/VScode.C++/Python/myenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|█████████████████████████████████████████| 207k/207k [00:00<00:00, 255kB/s]

Extracting files...
Path to dataset files: /Users/yuvrajbhatkariya/.cache/kagglehub/datasets/ashfakyeafi/spam-email-classification/versions/3


In [5]:
import pandas as pd
import tensorflow as tf
import tensorflow_hub as hub
import tensorflow_text as text

In [6]:
csv_file_path = os.path.join(path, 'email.csv')
df = pd.read_csv(csv_file_path)
df.head()

,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5573 entries, 0 to 5572
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Category  5573 non-null   object
 1   Message   5573 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB


In [8]:
import os
files_in_directory = os.listdir(path)
print("Files in the directory:", files_in_directory)

Files in the directory: ['email.csv']


In [9]:
df.isnull().sum()

Category    0
Message     0
dtype: int64

In [10]:
df['Category'].unique()

array(['ham', 'spam', '{"mode":"full"'], dtype=object)

In [11]:
df.sample(5)

,Category,Message
1304,ham,I cant pick the phone right now. Pls send a me...
1465,ham,Wat makes u thk i'll fall down. But actually i...
4472,ham,Wa... U so efficient... Gee... Thanx...
1740,ham,U guys never invite me anywhere :(
2737,ham,Really? I crashed out cuddled on my sofa.


In [12]:
bad_rows = df[df['Category'] == '{"mode":"full"']
print(bad_rows.index)

Index([5572], dtype='int64')


In [13]:
df.iloc[5572]

Category     {"mode":"full"
Message     isActive:false}
Name: 5572, dtype: object

In [14]:
df = df[df['Category'] != '{"mode":"full"']
df["Category"].unique()

array(['ham', 'spam'], dtype=object)

In [15]:
df.head()

,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


### Cheecking if data is balance or not: -

In [16]:
df_spam = df[df['Category'] == 'spam']
df_spam.shape

(747, 2)

In [17]:
df_ham = df[df['Category'] == 'ham']
df_ham.shape

(4825, 2)

In [18]:
len(df)

5572

### Oversampling spam upto 1400

In [19]:
dup_spam_rows = df_spam.sample(n = 700,random_state=42)
dup_spam_rows.sample(5)

,Category,Message
4514,spam,Money i have won wining number 946 wot do i do...
2730,spam,Urgent! Please call 09066612661 from your land...
576,spam,"You have won ?1,000 cash or a ?2,000 prize! To..."
2160,spam,FREE for 1st week! No1 Nokia tone 4 ur mob eve...
1469,spam,Hi its LUCY Hubby at meetins all day Fri & I w...


In [20]:
(dup_spam_rows.shape[0]) + (df_spam.shape[0])

1447

### Under sampling of ham : -

In [21]:
dup_ham_rows = df_ham.sample(n = 1447, random_state=42, replace=True)
dup_ham_rows.sample(5)

,Category,Message
5259,ham,Can help u swoop by picking u up from wherever...
1326,ham,Yeah jay's sort of a fucking retard
4994,ham,"HEY KATE, HOPE UR OK... WILL GIVE U A BUZ WEDL..."
3357,ham,Ok not a problem will get them a taxi. C ing ...
3584,ham,I sent your maga that money yesterday oh.


### Make a new data frame : -

In [22]:
new_spam_df = pd.concat([dup_spam_rows, df_spam], ignore_index=True)
new_spam_df.head()

,Category,Message
0,spam,Summers finally here! Fancy a chat or flirt wi...
1,spam,This is the 2nd time we have tried 2 contact u...
2,spam,Get ur 1st RINGTONE FREE NOW! Reply to this ms...
3,spam,Ur cash-balance is currently 500 pounds - to m...
4,spam,Last Chance! Claim ur £150 worth of discount v...


In [23]:
new_spam_df.shape

(1447, 2)

In [24]:
new_df = pd.concat([new_spam_df,dup_ham_rows])
new_df.sample(6)

,Category,Message
776,spam,Today's Offer! Claim ur £150 worth of discount...
1285,spam,thesmszone.com lets you send free anonymous an...
2506,ham,Congrats kano..whr s the treat maga?
670,spam,No. 1 Nokia Tone 4 ur mob every week! Just txt...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
2999,ham,No b4 Thursday


In [25]:
new_df.shape

(2894, 2)

In [26]:
df_spam = new_df[new_df['Category'] == 'spam']
print(df_spam.shape)
df_spam = new_df[new_df['Category'] == 'spam']
print(df_spam.shape)

(1447, 2)
(1447, 2)


## **Now here is the encoding part and also here we use BERT for converting text into vector**